# Chapter 4 — The matched filter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/04_matched_filter.ipynb)

**Goal.** A satellite's stars occupy a narrow region of the CMD (Chapter 2), while field stars spread over all of
it. Selecting or *weighting* stars by how much more likely they are under the satellite's CMD than under the
field's raises the contrast of the satellite. This is the **matched filter**, used for the tidal tails of Pal 5
(Rockosi et al. 2002, https://ui.adsabs.harvard.edu/abs/2002AJ....124..349R) and in most satellite searches since.

**You will learn**
1. three ways to select stars: a colour–magnitude box, an isochrone mask and continuous weights $w = s/b$;
2. a signal-to-noise statistic for weighted counts, with the background variance included;
3. to measure the gain of a filter on **injected** satellites, rather than on the object used to design it;
4. to apply the same filter to a stream: the tidal tails of Pal 5.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')
iso = load('isochrones_sdss')

# CMD grid shared by all filters: colour (g-r)_0 and magnitude g_0
cb, mb = np.arange(-0.4, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')


def cmd_index(c, m):
    """Grid cell of each star; -1 outside the grid."""
    i, j = np.digitize(c, cb) - 1, np.digitize(m, mb) - 1
    ok = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm))
    return np.where(ok, i * len(mm) + j, -1)


def satellite_cmd(mu, logage=10.1, feh=-2.1, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0), intrinsic=0.03):
    """Normalised satellite CMD density on the grid: isochrone at distance modulus mu, sampled uniformly along
    its length, weighted by the IMF, blurred by photometric errors (colour error as a function of g) and an
    intrinsic width. The default error law approximates SDSS (Chapter 0)."""
    I = iso[np.isclose(iso.logage, logage) & np.isclose(iso.feh, feh) & (iso.stage <= 7)].sort_values('m_ini')
    c, g = (I.g - I.r).to_numpy(), I.g.to_numpy() + mu
    arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(c), np.diff(g)))])
    s = np.linspace(0, arc[-1], 5000)
    pc, pg, w = np.interp(s, arc, c), np.interp(s, arc, g), np.gradient(np.interp(s, arc, I.int_imf))
    sc = np.hypot(np.sqrt(2) * err(pg), intrinsic)
    sm = np.hypot(err(pg), 0.1)
    S = np.zeros(CC.shape)
    for k in range(0, len(s), 500):
        sl = slice(k, k + 500)
        S += np.einsum('k,kij->ij', w[sl], np.exp(-0.5 * ((CC - pc[sl, None, None]) / sc[sl, None, None])**2
                                                      - 0.5 * ((MM - pg[sl, None, None]) / sm[sl, None, None])**2)
                       / sc[sl, None, None] / sm[sl, None, None])
    return S / S.sum()


def field_cmd(c, m, smooth=1.0):
    """Normalised field CMD density on the grid, from the stars of a control region (smoothed histogram).
    The density is floored at one star per cell: where the control region has no stars we do not know b, and
    an unfloored s/b would put enormous weight on (nearly) empty cells."""
    H = gaussian_filter(np.histogram2d(c, m, [cb, mb])[0], smooth)
    return np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()

## 4.1 Three filters for Boötes I

For each star $i$ we define a weight $w_i$ from its position in the CMD:

- **box**: $w = 1$ inside $0 < (g-r)_0 < 0.7$, $19 < g_0 < 22.5$ (the selection of Chapter 0), else 0;
- **isochrone mask**: $w = 1$ where the satellite density $s$ is above a threshold enclosing 90% of the satellite's
  stars, else 0;
- **matched filter**: $w = s/b$, the ratio of the satellite and field CMD densities (normalised to a maximum of 1).

The statistic is the weighted excess over the background, divided by its expected noise. With $N$ stars in an
on-region and a background predicted from an off-region of $\alpha^{-1}$ times its area,

$$ S = \sum_{\rm on} w_i - \alpha\sum_{\rm off} w_i, \qquad \sigma^2 \simeq \alpha\sum_{\rm off} w_i^2, \qquad {\rm S/N} = S/\sigma. $$

The variance is that of a weighted Poisson sum (each background star contributes $w_i^2$); the off-region term is
neglected because the off-region is ~100 times larger. The weights matter through the ratio of the signal they
keep to the $\sqrt{\sum w^2}$ of the background they admit — which is why $w = s/b$ is optimal for a weak signal.
One practical detail: $b$ is estimated from a finite control sample, so in sparsely populated parts of the CMD it is
poorly known; we floor it at one control star per cell, which caps the weights there (see `field_cmd`).

In [ ]:
ref = lvdb.loc['bootes_1']
cat = load('bootes1_sdss')
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
R = np.hypot(*tangent(cat.ra, cat.dec, ref.ra, ref.dec)) * 60
off = (R > 60) & (R < 120)
k_star = cmd_index(c0, g0)

S_sat = satellite_cmd(ref.distance_modulus)
B_fld = field_cmd(c0[off], g0[off])
level = np.sort(S_sat.ravel())[::-1][np.searchsorted(np.cumsum(np.sort(S_sat.ravel())[::-1]), 0.9)]
filters = {
    'box': ((CC > 0) & (CC < 0.7) & (MM > 19) & (MM < 22.5)).astype(float),
    'isochrone mask': (S_sat >= level).astype(float),
    'matched filter s/b': (S_sat / B_fld) / (S_sat / B_fld).max(),
}


def weights(F, k):
    """Weight of each star (index k into the CMD grid) under filter F."""
    return np.where(k >= 0, F.ravel()[np.maximum(k, 0)], 0.0)


def snr(w, on, off, alpha):
    return (w[on].sum() - alpha * w[off].sum()) / np.sqrt(alpha * (w[off]**2).sum())


on = R < ref.rhalf
alpha = ref.rhalf**2 / (120**2 - 60**2)
for name, F in filters.items():
    print(f'{name:20s}: S/N of Boötes I inside r_h = {snr(weights(F, k_star), on, off, alpha):5.1f}')

fig, ax = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)
ext = [cb[0], cb[-1], mb[-1], mb[0]]
for a, (t, F) in zip(ax, [('satellite s (μ = 19.11)', S_sat), ('field b', B_fld)] + list(filters.items())[1:]):
    a.imshow(F.T**0.3, extent=ext, aspect='auto', cmap='Greys')            # power 0.3 shows faint features too
    a.set(title=t, xlabel='$(g-r)_0$')
ax[0].set(ylabel='$g_0$')
plt.tight_layout()

**Figure 4.1.** The satellite CMD density $s$ (a blurred old, metal-poor isochrone at the distance of Boötes I),
the field density $b$ (from the annulus 60′–120′), the isochrone mask and the matched-filter weights $s/b$ (all
shown on a compressed scale, value$^{0.3}$). The weights are largest on the **blue horizontal branch** (BHB), where
satellite stars are expected and field stars are rare, and next along the blue edge of the giant branch.

## 4.2 Measuring the gain on injected satellites

Comparing the filters on Boötes I itself is biased: the box was chosen by looking at Boötes I. A fair test injects
synthetic satellites whose stars are drawn from the satellite CMD model into the *real* field, and compares the
filters on them. We place 200 satellites at random positions in the annulus 60′–120′ (away from Boötes I), each with
40 stars inside a 10′ radius, uniformly distributed in that circle, and compute the S/N of each filter using the real
field stars in the same circle plus the injected ones. The background is predicted from the rest of the annulus.

In [ ]:
rng = np.random.default_rng(4)
xi, eta = tangent(cat.ra, cat.dec, ref.ra, ref.dec)
pcell = S_sat.ravel() / S_sat.sum()
n_inj, r_ap = 40, 10 / 60
gains = {k: [] for k in filters}
for t in range(200):
    rad, ang = rng.uniform(1.25, 1.75), rng.uniform(0, 2 * np.pi)          # centre well inside the annulus
    x0, y0 = rad * np.cos(ang), rad * np.sin(ang)
    on_t = np.hypot(xi - x0, eta - y0) < r_ap
    off_t = off & ~on_t
    alpha_t = (np.pi * r_ap**2) / (np.pi * (2**2 - 1**2) - np.pi * r_ap**2)
    k_inj = rng.choice(pcell.size, n_inj, p=pcell)                           # CMD cells of the injected stars
    sn = {}
    for name, F in filters.items():
        w_real = weights(F, k_star)
        signal = w_real[on_t].sum() + F.ravel()[k_inj].sum() - alpha_t * w_real[off_t].sum()
        sn[name] = signal / np.sqrt(alpha_t * (w_real[off_t]**2).sum())
    for name in filters:
        gains[name].append(sn[name])
sn_arr = {k: np.array(v) for k, v in gains.items()}
gain = sn_arr['matched filter s/b'] / sn_arr['box']
boot = np.array([rng.choice(gain, gain.size).mean() for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print({k: round(float(np.median(v)), 1) for k, v in sn_arr.items()})
print(f'gain of matched filter over box: mean {gain.mean():.2f}, 95% interval [{lo:.2f}, {hi:.2f}]')

fig, ax = plt.subplots(figsize=(6.5, 4))
for name, v in sn_arr.items():
    ax.hist(v, np.linspace(-2, 14, 33), histtype='step', lw=2, label=f'{name}: median {np.median(v):.1f}')
ax.set(xlabel='S/N of an injected 40-star satellite', ylabel='injections')
ax.legend(fontsize=8)
plt.tight_layout()
assert lo > 1

**Figure 4.2.** S/N of 200 injected satellites under the three filters. The isochrone mask beats the box, and the
continuous weights beat both, on the same stars. The gain is not a free lunch. Most of it comes from the BHB: the
PARSEC model puts all horizontal-branch stars of an old, metal-poor population on the blue side, as in Boötes I,
but satellites with red horizontal branches exist, and for them the gain is smaller. In general the gain holds only
if the satellite resembles the assumed isochrone and distance; a search must therefore scan the distance
(Chapter 5) and, ideally, the population.

## 4.3 The same filter on a stream: Pal 5

The globular cluster Pal 5 (μ = 16.7, ~22 kpc) is being torn apart by the Milky Way's tides; its stars form two
tails more than 20° long (Odenkirchen et al. 2001; Rockosi et al. 2002). Pal 5 is younger and more metal-rich than
Boötes I, so we use log age = 10.05 and [Fe/H] = −1.3. We weight every star of the 22° × 16° field by $s/b$, with $b$
from the whole field (the stream is a tiny fraction of it), and map the summed weights in 0.1° cells, divided by
the fraction of each cell that SDSS observed.

In [ ]:
p5 = lvdb.loc['palomar_05']
pal = load('pal5_sdss')
cov = load('pal5_coverage')
pg0 = (pal.psfmag_g - pal.extinction_g).to_numpy()
pc0 = pg0 - (pal.psfmag_r - pal.extinction_r).to_numpy()
kp = cmd_index(pc0, pg0)
S5 = satellite_cmd(p5.distance_modulus, logage=10.05, feh=-1.3)
B5 = field_cmd(pc0, pg0)
F5 = {'box': ((CC > 0.1) & (CC < 0.5) & (MM > 19.5) & (MM < 22.5)).astype(float),
      'matched filter s/b': (S5 / B5) / (S5 / B5).max()}
ra_e, dec_e = np.arange(222, 244.01, 0.1), np.arange(-8, 8.01, 0.1)
observed = np.zeros((len(ra_e) - 1, len(dec_e) - 1), bool)
i, j = cov.ira - 2220, cov.idec + 80
ok = (i >= 0) & (i < observed.shape[0]) & (j >= 0) & (j < observed.shape[1]) & (cov.n > 20)
observed[i[ok], j[ok]] = True

fig, ax = plt.subplots(1, 2, figsize=(15, 5.6))
maps = {}
for a, (name, F) in zip(ax, F5.items()):
    w = weights(F, kp)
    H = np.histogram2d(pal.ra, pal.dec, [ra_e, dec_e], weights=w)[0]
    H = np.where(observed, gaussian_filter(H, 1.5) / np.maximum(gaussian_filter(observed.astype(float), 1.5), 1e-3), np.nan)
    maps[name] = H
    v = np.nanpercentile(H, [2, 99.5])
    a.imshow(H.T, origin='lower', extent=[ra_e[0], ra_e[-1], dec_e[0], dec_e[-1]], cmap='Greys', vmin=v[0], vmax=v[1])
    a.set(xlabel='RA [deg]', ylabel='Dec [deg]', title=name, xlim=(244, 222))
plt.tight_layout()

**Figure 4.3.** Smoothed star counts in the Pal 5 field with a simple box selection (left) and with matched-filter
weights (right). Regions outside the SDSS footprint are blank. Pal 5 sits at (229.0, −0.1); its two tails run
north-east to RA ≈ 234° and south-west to RA ≈ 226°, and are sharper with the matched filter. The second blob, at
(229.6, 2.1), is the globular cluster M5 (7.5 kpc): its stars are brighter than Pal 5's, but its main sequence crosses
the filter. The dark north-eastern corner is the rising density of the Galactic disc and bulge towards lower
latitude. Chapter 7 measures the stream's track and significance.

## Summary

- The S/N of a weighted count is $S/\sqrt{\alpha\sum_{\rm off} w^2}$; weights $s/b$ maximise it for a faint signal.
- Judge a filter on injected objects, not on the one that inspired it.
- The same filter reveals the tidal tails of Pal 5, a stream spanning the whole field.